[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/09_tag_11_12_gru_update_reset_gate.ipynb)

# Reset- und Update-Signal einer GRU

## Lernziele und Vorgehensweise

Eine **Gated Recurrent Unit (GRU)** steuert mit zwei Gates, wie alte und neue Information zusammenwirken. Nach diesem Notebook kannst du erklären,

1. wie das **Reset-Signal** $r_t$ den Einfluss des vorherigen Hidden States auf den Kandidaten $g_t$ verändert,
2. wie das **Update-Signal** $z_t$ den alten Hidden State $h_{t-1}$ und den Kandidaten $g_t$ zum neuen Hidden State $h_t$ mischt,
3. weshalb Signalwerte nahe 0 und nahe 1 sehr unterschiedliches Verhalten erzeugen und
4. wie sich dieselbe Rechnung über mehrere Zeitschritte fortsetzt.

Im Hauptbeispiel wird **kein Modell trainiert**. Echte GRUs berechnen $r_t$, $g_t$ und $z_t$ aus Eingaben, Hidden States und gelernten Gewichten. Hier geben wir die beiden Signalwerte $r_t$ und $z_t$ absichtlich vor. So können wir ihre Wirkung isolieren, alle Zwischenschritte kontrollieren und Ursache und Wirkung eindeutig zuordnen. Das Lernziel ist Transparenz, nicht Vorhersageleistung.


## Technische Vorbereitung

Wir benötigen nur NumPy für die Rechnung, Pandas für Tabellen und Matplotlib für die Diagramme. Ein fester Zufallsstartwert dokumentiert die Reproduzierbarkeit, obwohl die folgenden Beispiele vollständig deterministisch sind. Einheitliche Diagrammeinstellungen machen die getrennten Szenarien später direkt vergleichbar.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Feste Einstellungen halten Darstellung und eventuelle Zufallsoperationen reproduzierbar.
np.random.seed(42)
pd.set_option("display.precision", 4)
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})


## Grundidee und verwendete GRU-Konvention

Zu einem Zeitschritt $t$ erhält die GRU die aktuelle Eingabe $x_t$ und den vorherigen Hidden State $h_{t-1}$. Das Reset-Signal $r_t$ wirkt **innerhalb der Kandidatenberechnung**; das Update-Signal $z_t$ wirkt anschließend **bei der Mischung zum neuen Hidden State**.

In diesem Notebook gilt durchgehend genau folgende Konvention:

\begin{align}
r_t &= \sigma\!\left(W_r[h_{t-1},x_t]+b_r\right) \\
g_t &= \tanh\!\left(W_g[x_t,r_t\odot h_{t-1}]+b_g\right) \\
z_t &= \sigma\!\left(W_z[x_t,h_{t-1}]+b_z\right) \\
h_t &= z_t\odot h_{t-1} + (1-z_t)\odot g_t
\end{align}

**Reset-Signal:** Bei $r_t$ nahe 0 wird der alte Hidden State in der Kandidatenberechnung weitgehend ausgeblendet. Bei $r_t$ nahe 1 fließt er stark in $g_t$ ein. Das Reset-Signal entscheidet also nicht direkt, wie viel des alten Zustands im endgültigen $h_t$ bleibt.

**Update-Signal:** Bei $z_t$ nahe 1 dominiert $z_t\odot h_{t-1}$, also der alte Hidden State. Bei $z_t$ nahe 0 dominiert $(1-z_t)\odot g_t$, also der neue Kandidat. Ein hoher Wert von $z_t$ bewahrt somit die bisherige Information; ein kleiner Wert aktualisiert den Zustand stärker.

Die Parallele zu Quellen mit der umgekehrten Update-Konvention ist direkt: Dort heißt häufig $h_t=(1-z_t)h_{t-1}+z_t\tilde h_t$. Das dortige $\tilde h_t$ entspricht unserem $g_t$, und das dortige Update Gate entspricht $1-z_t$ in diesem Notebook. Hier gilt durchgehend: **großes $z_t$ behält alt, kleines $z_t$ übernimmt stärker den Kandidaten**.

Der neue Hidden State wird an den nächsten Zeitschritt und gegebenenfalls an die nächste GRU-Schicht weitergegeben: $h_{t+1}=\operatorname{GRU}(x_{t+1},h_t)$. Für eine konkrete Vorhersage folgt meist noch eine Ausgabeschicht, zum Beispiel $\hat y_t=g_{\mathrm{out}}(W_yh_t+b_y)$.

Sigmoid-Funktionen liefern für endliche Eingaben Werte strikt zwischen 0 und 1. Exakt 0 oder 1 werden normalerweise nicht erreicht. Deshalb stehen in den kontrollierten Szenarien **0,05** und **0,95** für „nahe 0“ und „nahe 1“.


## Feste Parameter und eine gemeinsame Berechnungsfunktion

Für den einzelnen GRU-Schritt verwenden wir $x_t=0{,}8$, $h_{t-1}=-0{,}6$, $W_x=1{,}0$, $W_h=1{,}2$ und $b_g=0{,}0$. Zur transparenten Rechnung zerlegen wir $W_g[x_t,r_t\odot h_{t-1}]$ in $W_xx_t+W_h(r_t\odot h_{t-1})$. Die Funktion gibt bewusst nicht nur $h_t$, sondern auch alle didaktisch wichtigen Zwischenergebnisse zurück. Dadurch können wir Kandidatenbildung und Update-Mischung getrennt untersuchen.

Alle Größen sind eindimensionale NumPy-Arrays. Die elementweisen Produkte entsprechen in diesem vereinfachten Fall der skalaren GRU-Rechnung.


In [ ]:
# Feste Parameter des transparenten Einzelbeispiels, jeweils mit Hidden-Dimension 1.
x_t = np.array([0.8], dtype=float)
h_vorher = np.array([-0.6], dtype=float)
W_x = np.array([1.0], dtype=float)
W_h = np.array([1.2], dtype=float)
b_g = np.array([0.0], dtype=float)


def gru_schritt(x_aktuell, h_alt, r_t, z_t,
               eingabegewicht, rekurrentes_gewicht, bias):
    '''Berechnet einen transparenten GRU-Schritt mit allen Zwischenwerten.'''
    x_aktuell = np.asarray(x_aktuell, dtype=float)
    h_alt = np.asarray(h_alt, dtype=float)
    r_t = np.asarray(r_t, dtype=float)
    z_t = np.asarray(z_t, dtype=float)
    eingabegewicht = np.asarray(eingabegewicht, dtype=float)
    rekurrentes_gewicht = np.asarray(rekurrentes_gewicht, dtype=float)
    bias = np.asarray(bias, dtype=float)

    # Beide Signale und der Hidden State müssen komponentenweise zueinander passen.
    assert x_aktuell.shape == h_alt.shape == r_t.shape == z_t.shape
    assert np.all((0.0 < r_t) & (r_t < 1.0))
    assert np.all((0.0 < z_t) & (z_t < 1.0))

    # Das Reset-Signal dämpft den alten Hidden State nur für die Berechnung von g_t.
    r_t_mal_h_alt = r_t * h_alt

    # Aktuelle Eingabe und resettete Erinnerung bilden gemeinsam die Voraktivierung.
    voraktivierung = (
        eingabegewicht * x_aktuell
        + rekurrentes_gewicht * r_t_mal_h_alt
        + bias
    )

    # tanh begrenzt den neuen Kandidatenzustand auf den Bereich von -1 bis 1.
    g_t = np.tanh(voraktivierung)

    # Hohes z_t bewahrt den alten Zustand; (1-z_t) gewichtet den neuen Kandidaten g_t.
    anteil_alt = z_t * h_alt
    anteil_neu = (1.0 - z_t) * g_t

    # Der neue Hidden State ist exakt die Summe der beiden Update-Beiträge.
    h_neu = anteil_alt + anteil_neu

    return {
        "r_t_mal_h_alt": r_t_mal_h_alt,
        "voraktivierung": voraktivierung,
        "g_t": g_t,
        "anteil_alt": anteil_alt,
        "anteil_neu": anteil_neu,
        "h_neu": h_neu,
    }


## Dimensionen des Einzelbeispiels

Die Hidden-State-Dimension beträgt zunächst 1. Deshalb bestehen Reset-Signal $r_t$, Update-Signal $z_t$, Kandidat $g_t$ und Hidden State $h_t$ jeweils aus genau einer Komponente. Reale GRUs verwenden meist Hidden States mit vielen Komponenten; dann sind auch beide Signale und der Kandidat Vektoren derselben Hidden-Dimension.

Die Skalaridee wird hier trotzdem als Array der Form `(1,)` umgesetzt. Damit bleibt der Rechenweg leicht nachvollziehbar und die Datenstruktur erinnert bereits an den Vektorfall. Die folgende Probe führt einen Schritt aus, gibt alle geforderten Formen tatsächlich aus und prüft sie mit Assertions.


In [ ]:
r_probe = np.array([0.95])
z_probe = np.array([0.95])
probe = gru_schritt(
    x_t, h_vorher, r_probe, z_probe, W_x, W_h, b_g
)

# Die Ausgabe dokumentiert, dass auch das scheinbar skalare Beispiel Vektoren nutzt.
formen = {
    "x_t": x_t.shape,
    "h_(t-1)": h_vorher.shape,
    "r_t": r_probe.shape,
    "z_t": z_probe.shape,
    "g_t": probe["g_t"].shape,
    "h_t": probe["h_neu"].shape,
}
for name, form in formen.items():
    print(f"{name}.shape = {form}")

assert all(form == (1,) for form in formen.values())
assert np.all(np.isfinite(np.concatenate(list(probe.values()))))


## Vier kontrollierte Szenarien

Wir kombinieren für $r_t$ und $z_t$ jeweils einen niedrigen Signalwert von 0,05 mit einem hohen Signalwert von 0,95. Die aktuelle Eingabe, der vorherige Hidden State und die Gewichte bleiben in allen vier Szenarien unverändert. Unterschiede können dadurch ausschließlich auf Reset- und Update-Signal zurückgeführt werden.

Eine gemeinsame Auswertungsfunktion verhindert doppelte Berechnungs- und Plotlogik. Sie zeigt pro Szenario alle Zwischenwerte und anschließend genau die drei Zustände, die den Informationsfluss besonders anschaulich machen: $h_{t-1}$, $g_t$ und $h_t$. Alle vier Einzelplots verwenden dieselben y-Achsengrenzen.


In [ ]:
szenario_definitionen = {
    "A": {"name": "Alten Zustand weitgehend behalten", "r": 0.95, "z": 0.95},
    "B": {"name": "Kandidaten mit Erinnerung übernehmen", "r": 0.95, "z": 0.05},
    "C": {"name": "Alte Information zurücksetzen und Kandidaten übernehmen", "r": 0.05, "z": 0.05},
    "D": {"name": "Kandidaten ohne alte Erinnerung bilden, aber alten Zustand behalten", "r": 0.05, "z": 0.95},
}


def als_zahl(array):
    '''Liest den einzigen Wert eines Arrays aus, ohne die Berechnung zu runden.'''
    return float(np.asarray(array).item())


# Alle Szenarien nutzen dieselbe fachliche Funktion; gespeichert bleiben ungerundete Werte.
szenario_ergebnisse = {}
for kennung, definition in szenario_definitionen.items():
    szenario_ergebnisse[kennung] = gru_schritt(
        x_t,
        h_vorher,
        np.array([definition["r"]]),
        np.array([definition["z"]]),
        W_x,
        W_h,
        b_g,
    )


def szenario_zeile(kennung):
    '''Bereitet alle Zwischenwerte eines Szenarios für eine Tabelle auf.'''
    definition = szenario_definitionen[kennung]
    ergebnis = szenario_ergebnisse[kennung]
    return {
        "Szenario": kennung,
        "Reset-Signal r_t": definition["r"],
        "Update-Signal z_t": definition["z"],
        "Hidden State h_(t-1)": als_zahl(h_vorher),
        "r_t * h_(t-1)": als_zahl(ergebnis["r_t_mal_h_alt"]),
        "Voraktivierung g_t": als_zahl(ergebnis["voraktivierung"]),
        "Kandidat g_t": als_zahl(ergebnis["g_t"]),
        "alter Anteil z_t * h_(t-1)": als_zahl(ergebnis["anteil_alt"]),
        "neuer Anteil (1-z_t) * g_t": als_zahl(ergebnis["anteil_neu"]),
        "Hidden State h_t": als_zahl(ergebnis["h_neu"]),
    }


def zeige_szenario(kennung):
    '''Zeigt Zwischenwerte und einen vergleichbar skalierten Einzelplot.'''
    definition = szenario_definitionen[kennung]
    ergebnis = szenario_ergebnisse[kennung]
    display(pd.DataFrame([szenario_zeile(kennung)]).round(4))

    bezeichnungen = ["Hidden State\nh_(t-1)", "Kandidat\ng_t", "Hidden State\nh_t"]
    werte = [
        als_zahl(h_vorher),
        als_zahl(ergebnis["g_t"]),
        als_zahl(ergebnis["h_neu"]),
    ]
    fig, ax = plt.subplots()
    balken = ax.bar(
        bezeichnungen,
        werte,
        color=["#4C78A8", "#F58518", "#54A24B"],
        label="Zustandswert",
    )
    ax.axhline(0, color="black", linewidth=0.9)
    ax.set_ylim(-0.75, 0.75)
    ax.set_xlabel("Zustandsgröße")
    ax.set_ylabel("Wert")
    ax.set_title(
        f"Szenario {kennung}: r = {definition['r']:.2f}, z = {definition['z']:.2f}"
    )
    ax.legend()
    ax.bar_label(balken, fmt="%.3f", padding=3)
    plt.tight_layout()
    plt.show()


### Szenario A – Alten Zustand weitgehend behalten

Mit $r_t=0{,}95$ beeinflusst der alte Hidden State den Kandidaten deutlich: Aus $-0{,}6$ wird im Kandidatenpfad zunächst $-0{,}57$. Zusammen mit der Eingabe entsteht die Voraktivierung $0{,}116$ und damit der Kandidat $g_t\approx0{,}115$.

Das große Update-Signal $z_t=0{,}95$ bewahrt $-0{,}570$ aus dem alten Zustand. Der Gegenwert $1-z_t=0{,}05$ übernimmt vom Kandidaten nur etwa $0{,}006$, sodass $h_t\approx-0{,}564$ nahe am alten Zustand bleibt. Für den Endeffekt ist hier hauptsächlich das **große Update-Signal** verantwortlich.


In [ ]:
# Szenario A separat ausgeben und mit der gemeinsamen Achsenskalierung visualisieren.
zeige_szenario("A")


### Szenario B – Kandidaten mit Erinnerung übernehmen

Das Reset-Signal ist wie in A $r_t=0{,}95$. Deshalb entstehen dieselbe Voraktivierung $0{,}116$ und derselbe Kandidat $g_t\approx0{,}115$. Das Reset-Signal verändert sich zwischen A und B nicht.

Nun ist jedoch $z_t=0{,}05$: Der alte Zustand trägt nur $-0{,}030$ bei, während $1-z_t=0{,}95$ etwa $0{,}110$ aus $g_t$ übernimmt. Der neue Hidden State wird dadurch $h_t\approx0{,}080$. Der starke Wechsel gegenüber A stammt **ausschließlich vom kleinen Update-Signal**, das den Kandidaten weitgehend übernimmt.


In [ ]:
# Szenario B nutzt dieselbe Berechnung und erhält trotzdem einen eigenen Plot.
zeige_szenario("B")


### Szenario C – Alte Information zurücksetzen und neuen Kandidaten übernehmen

Mit $r_t=0{,}05$ gelangen vom alten Hidden State nur $-0{,}030$ in die Kandidatenberechnung. Die aktuelle Eingabe dominiert nun: Die Voraktivierung steigt auf $0{,}764$ und der Kandidat auf $g_t\approx0{,}643$. Der Vergleich mit B zeigt besonders klar die Wirkung des **Reset-Signals auf den Kandidaten**.

Das kleine Update-Signal $z_t=0{,}05$ bewahrt nur $-0{,}030$ aus dem alten Zustand; der Gegenwert $1-z_t=0{,}95$ übernimmt etwa $0{,}611$ aus $g_t$. So entsteht $h_t\approx0{,}581$. Reset- und Update-Signal haben hier verschiedene, aufeinanderfolgende Aufgaben.


In [ ]:
# Szenario C macht den Einfluss des kleinen Reset-Signals separat sichtbar.
zeige_szenario("C")


### Szenario D – Kandidaten ohne alte Erinnerung bilden, aber alten Zustand behalten

Wegen $r_t=0{,}05$ entstehen wie in C die Voraktivierung $0{,}764$ und der Kandidat $g_t\approx0{,}643$. Der Kandidat unterscheidet sich also deutlich von A und B, weil der negative alte Zustand in seiner Berechnung fast ausgeblendet wird.

Trotzdem bleibt der endgültige Hidden State mit $h_t\approx-0{,}538$ nahe bei $-0{,}6$: Das große $z_t=0{,}95$ bewahrt als alten Beitrag $-0{,}570$ und der Gegenwert $1-z_t=0{,}05$ übernimmt nur etwa $0{,}032$ aus $g_t$. Dieses Szenario zeigt: **Ein kleines Reset-Signal löscht den endgültigen Hidden State nicht automatisch.** Die direkte Mischung bestimmt das Update-Signal.


In [ ]:
# Szenario D schließt die getrennte Analyse der vier Signalkombinationen ab.
zeige_szenario("D")


## Gemeinsame Tabelle und kompakter Szenarienvergleich

Erst nach den vier getrennten Analysen führen wir die Ergebnisse zusammen. Die Tabelle enthält alle verlangten Beiträge. Gerundet wird nur die Anzeige; in `vergleichstabelle` bleiben die ursprünglichen Gleitkommawerte erhalten.

Als Kontrollpaar gilt: A und B haben denselben Kandidaten $g_t$, weil ihr Reset-Signal gleich ist. C und D haben ebenfalls denselben Kandidaten. Innerhalb dieser Paare verändert erst das Update-Signal den neuen Hidden State.


In [ ]:
# Die ungerundete Tabelle bleibt die Grundlage aller Prüfungen und Diagramme.
vergleichstabelle = pd.DataFrame(
    [szenario_zeile(kennung) for kennung in szenario_definitionen]
)
display(vergleichstabelle.round(4))

# Form-, Bereichs- und Endlichkeitsprüfungen sichern die fachliche Passung.
assert len(vergleichstabelle) == 4
assert vergleichstabelle["Hidden State h_t"].notna().all()
assert np.isfinite(vergleichstabelle.select_dtypes(include=[np.number]).to_numpy()).all()
assert vergleichstabelle["Reset-Signal r_t"].between(0.0, 1.0, inclusive="neither").all()
assert vergleichstabelle["Update-Signal z_t"].between(0.0, 1.0, inclusive="neither").all()
assert np.isclose(
    vergleichstabelle.loc[0, "Kandidat g_t"],
    vergleichstabelle.loc[1, "Kandidat g_t"],
)
assert np.isclose(
    vergleichstabelle.loc[2, "Kandidat g_t"],
    vergleichstabelle.loc[3, "Kandidat g_t"],
)


**Interpretation direkt unter der Tabelle:**

- **A:** Das hohe Reset-Signal führt zu $g_t=0{,}115$; das hohe Update-Signal bewahrt mit $z_t=0{,}95$ vor allem den alten Beitrag $-0{,}570$. Nur $0{,}006$ aus $g_t$ fließen in $h_t$ ein.
- **B:** $g_t$ bleibt wegen desselben Reset-Signals $0{,}115$. Das kleine Update-Signal tauscht aber die Gewichte der Beiträge: $-0{,}030$ alt und $0{,}110$ neu ergeben $0{,}080$.
- **C:** Das niedrige Reset-Signal verändert $g_t$ auf $0{,}643$, weil die negative Erinnerung kaum noch gegen die positive Eingabe wirkt. Das kleine Update-Signal übernimmt diesen Kandidaten fast vollständig; $h_t$ wird $0{,}581$.
- **D:** $g_t$ ist wie in C $0{,}643$, wird beim hohen Update-Signal aber kaum genutzt. Deshalb bleibt $h_t=-0{,}538$ trotz kleinem Reset-Signal nahe am alten Zustand.

Der Kandidat $g_t$ wird somit hauptsächlich durch das **Reset-Signal** verändert, der endgültige Hidden State durch die anschließende **Update-Mischung**.


In [ ]:
# Der Vergleichsplot zeigt nur den Endwert und ergänzt die vier Einzelplots.
fig, ax = plt.subplots()
farben = ["#4C78A8", "#F58518", "#54A24B", "#E45756"]
balken = ax.bar(
    vergleichstabelle["Szenario"],
    vergleichstabelle["Hidden State h_t"],
    color=farben,
    label="neuer Hidden State",
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_ylim(-0.75, 0.75)
ax.set_xlabel("Szenario")
ax.set_ylabel("$h_t$")
ax.set_title("Kompakter Vergleich der neuen Hidden States")
ax.legend()
ax.bar_label(balken, fmt="%.3f", padding=3)
plt.tight_layout()
plt.show()


## Visuelle Zerlegung des Update-Signals

Die Gleichung $h_t=z_t\odot h_{t-1}+(1-z_t)\odot g_t$ enthält zwei vorzeichenbehaftete Beiträge. Deshalb verwenden wir getrennte Balken statt eines gewöhnlichen gestapelten Balkens: Negative alte Beiträge und positive neue Beiträge bleiben so eindeutig erkennbar. Die Rauten markieren jeweils ihre Summe, also den resultierenden Hidden State.


In [ ]:
# Getrennte Balken zeigen auch bei unterschiedlichen Vorzeichen beide Informationsquellen korrekt.
positionen = np.arange(len(vergleichstabelle))
breite = 0.34
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    positionen - breite / 2,
    vergleichstabelle["alter Anteil z_t * h_(t-1)"],
    breite,
    label=r"alter Anteil $z_t h_{t-1}$",
    color="#4C78A8",
)
ax.bar(
    positionen + breite / 2,
    vergleichstabelle["neuer Anteil (1-z_t) * g_t"],
    breite,
    label=r"neuer Anteil $(1-z_t)g_t$",
    color="#F58518",
)
ax.scatter(
    positionen,
    vergleichstabelle["Hidden State h_t"],
    marker="D",
    s=70,
    color="#54A24B",
    label=r"Summe $h_t$",
    zorder=3,
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(positionen, vergleichstabelle["Szenario"])
ax.set_ylim(-0.75, 0.75)
ax.set_xlabel("Szenario")
ax.set_ylabel("Beitrag zum Hidden State")
ax.set_title("Update-Signal: alter Beitrag + neuer Beitrag = Hidden State h_t")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


## Teil B – Verarbeitung einer vollständigen Sequenz

Nun wenden wir dieselbe Funktion Schritt für Schritt auf acht Eingaben an. Die skalare Folge erhält zunächst die Form `(8, 1)`: acht Zeitschritte mit je einem Merkmal. Für eine typische RNN-/GRU-Schnittstelle ergänzen wir vorne die Batchdimension und erhalten `(1, 8, 1)`.

Die drei Dimensionen bedeuten **(Batchgröße, Sequenzlänge, Anzahl Merkmale)**. Hier befinden sich also 1 Sequenz, 8 Zeitschritte und 1 Merkmal im Batch. Reset-Signal $r_t$ und Update-Signal $z_t$ werden erneut bewusst vorgegeben und besitzen jeweils die Form `(8, 1)`. Der initiale Hidden State ist $h_0=0$; danach wird jeder berechnete Zustand zum vorherigen Zustand des nächsten Schritts.


In [ ]:
# Acht skalare Eingaben werden als acht Zeilen mit je einem Merkmal gespeichert.
x_sequenz = np.array([0.2, 0.4, 0.7, -0.3, -0.8, 0.1, 0.9, 0.2], dtype=float).reshape(8, 1)

# Die zusätzliche erste Achse steht für genau eine Sequenz im Batch.
x_batch = x_sequenz.reshape(1, 8, 1)

# Vorgegebene Signale machen gezielte Kombinationen über die Zeit beobachtbar.
r_sequenz = np.array(
    [0.95, 0.95, 0.95, 0.95, 0.05, 0.05, 0.95, 0.95],
    dtype=float,
).reshape(8, 1)
z_sequenz = np.array(
    [0.80, 0.80, 0.70, 0.20, 0.05, 0.80, 0.10, 0.80],
    dtype=float,
).reshape(8, 1)
h_0 = np.array([0.0], dtype=float)

print(f"x_sequenz.shape = {x_sequenz.shape}")
print(f"x_batch.shape = {x_batch.shape}")
print(f"r_sequenz.shape = {r_sequenz.shape}")
print(f"z_sequenz.shape = {z_sequenz.shape}")

assert x_sequenz.shape == (8, 1)
assert x_batch.shape == (1, 8, 1)
assert r_sequenz.shape == z_sequenz.shape == (8, 1)
assert np.all((0.0 < r_sequenz) & (r_sequenz < 1.0))
assert np.all((0.0 < z_sequenz) & (z_sequenz < 1.0))


## Sequenz Schritt für Schritt berechnen

In jedem Schleifendurchlauf wird zuerst der vorherige Hidden State gesichert. Danach berechnet `gru_schritt` das Produkt $r_t\odot h_{t-1}$, die Kandidatenvoraktivierung, $g_t$ und beide Update-Beiträge. Der neue Hidden State wird kopiert und dient im nächsten Durchlauf als $h_{t-1}$.

Wichtig ist diese zeitliche Abhängigkeit: Obwohl die Signalwerte vorgegeben sind, hängen spätere Kandidaten und Hidden States von allen zuvor berechneten Zuständen ab. Die Ergebnisliste speichert deshalb für jeden Zeitschritt sämtliche geforderten Größen ungerundet.


In [ ]:
sequenz_zeilen = []
h_aktuell = h_0.copy()

# Die GRU verarbeitet genau einen Eingabewert sowie r_t und z_t pro Zeitschritt.
for index in range(len(x_sequenz)):
    h_vor_schritt = h_aktuell.copy()
    ergebnis = gru_schritt(
        x_sequenz[index],
        h_vor_schritt,
        r_sequenz[index],
        z_sequenz[index],
        W_x,
        W_h,
        b_g,
    )

    # Alle Zwischenwerte werden gespeichert, damit die Tabelle den Rechenweg offenlegt.
    sequenz_zeilen.append({
        "t": index + 1,
        "x_t": als_zahl(x_sequenz[index]),
        "r_t": als_zahl(r_sequenz[index]),
        "z_t": als_zahl(z_sequenz[index]),
        "h_(t-1)": als_zahl(h_vor_schritt),
        "r_t * h_(t-1)": als_zahl(ergebnis["r_t_mal_h_alt"]),
        "Voraktivierung": als_zahl(ergebnis["voraktivierung"]),
        "g_t": als_zahl(ergebnis["g_t"]),
        "z_t * h_(t-1)": als_zahl(ergebnis["anteil_alt"]),
        "(1 - z_t) * g_t": als_zahl(ergebnis["anteil_neu"]),
        "h_t": als_zahl(ergebnis["h_neu"]),
    })

    # Nur der neue Hidden State wird als Erinnerung an den nächsten Schritt weitergereicht.
    h_aktuell = ergebnis["h_neu"].copy()

sequenz_tabelle = pd.DataFrame(sequenz_zeilen)

assert len(sequenz_tabelle) == 8
assert np.isfinite(sequenz_tabelle.select_dtypes(include=[np.number]).to_numpy()).all()
assert np.allclose(
    sequenz_tabelle["h_t"],
    sequenz_tabelle["z_t * h_(t-1)"]
    + sequenz_tabelle["(1 - z_t) * g_t"],
)


## Schrittweise Ergebnistabelle

Jede Zeile lässt sich von links nach rechts als vollständiger GRU-Schritt lesen. Zuerst dämpft $r_t$ den vorherigen Zustand für die Berechnung von $g_t$. Nach der `tanh`-Aktivierung gewichtet $z_t$ den alten Zustand und $1-z_t$ den neuen Kandidaten. Die letzten drei Spalten erfüllen in jeder Zeile die Gleichung „alter Beitrag + neuer Beitrag = $h_t$“.

Die Anzeige wird auf vier Nachkommastellen gerundet. Alle nachfolgenden Diagramme und Assertions verwenden weiterhin die ungerundeten Daten in `sequenz_tabelle`.


In [ ]:
# Nur die Darstellung wird gerundet; die gespeicherte Tabelle behält volle Genauigkeit.
display(sequenz_tabelle.round(4))


### Interpretation ausgewählter Zeitschritte

- **Hoher Reset-, hoher Update-Wert (t = 2, $r_t=0{,}95$, $z_t=0{,}80$):** Der alte Zustand $0{,}0395$ geht als $0{,}0375$ fast vollständig in die Kandidatenberechnung ein. Der Kandidat ist $g_2=0{,}4178$, aber der Gegenwert $1-z_t=0{,}20$ übernimmt davon nur $0{,}0836$. Zusammen mit $0{,}0316$ altem Anteil entsteht $h_2=0{,}1151$.
- **Hoher Reset-, niedriger Update-Wert (t = 7, $r_t=0{,}95$, $z_t=0{,}10$):** Vom negativen alten Zustand $-0{,}4871$ gehen $-0{,}4628$ in die Kandidatenberechnung ein. Die positive Eingabe $0{,}9$ führt dennoch zu $g_7=0{,}3316$. Der Gegenwert $1-z_t=0{,}90$ gewichtet den Kandidaten mit $0{,}2985$, während alt nur $-0{,}0487$ beiträgt; $h_7$ wechselt auf $0{,}2498$.
- **Niedriger Reset-, niedriger Update-Wert (t = 5, $r_t=0{,}05$, $z_t=0{,}05$):** Nur $0{,}0038$ des alten Zustands fließen in die Kandidatenberechnung. Die Eingabe $-0{,}8$ dominiert und erzeugt $g_5=-0{,}6615$. Der Gegenwert $1-z_t=0{,}95$ übernimmt davon $-0{,}6284$; mit dem kleinen alten Beitrag $0{,}0038$ folgt $h_5=-0{,}6245$.
- **Niedriger Reset-, hoher Update-Wert (t = 6, $r_t=0{,}05$, $z_t=0{,}80$):** Obwohl der alte Zustand $-0{,}6245$ in der Kandidatenberechnung auf $-0{,}0312$ gedämpft wird und $g_6$ dadurch leicht positiv ($0{,}0624$) ist, bleiben im Endzustand $-0{,}4996$ aus dem alten Pfad. Der neue Anteil trägt nur $0{,}0125$ bei; $h_6=-0{,}4871$. Ein kleines Reset-Signal allein löscht den Hidden State also nicht.

Die tatsächlichen Zahlen bestätigen die Aufgabenteilung: $r_t$ verändert zuerst den Kandidatenpfad; danach bewahrt $z_t$ den alten Zustand und $1-z_t$ übernimmt den neuen Kandidaten.


## Plot 1 – Eingabesequenz

Der erste Sequenzplot zeigt ausschließlich die acht Eingabewerte. Positive und negative Abschnitte sind so klar erkennbar, ohne sie bereits mit Signalen oder Zuständen zu vermischen.


In [ ]:
# Die Eingabe wird separat gezeigt, damit ihr Vorzeichenwechsel deutlich bleibt.
fig, ax = plt.subplots()
ax.plot(sequenz_tabelle["t"], sequenz_tabelle["x_t"], marker="o", label="$x_t$")
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(sequenz_tabelle["t"])
ax.set_xlabel("Zeitschritt t")
ax.set_ylabel("Eingabewert")
ax.set_title("Eingabesequenz über acht Zeitschritte")
ax.legend()
plt.tight_layout()
plt.show()


## Plot 2 – Reset-Signal und Update-Signal

Beide Signalverläufe liegen zwischen 0 und 1 und dürfen deshalb eine gemeinsame Achse verwenden. Die feste y-Achse von 0 bis 1 macht niedrige und hohe Signalphasen unmittelbar sichtbar.


In [ ]:
# Nur die beiden gleich skalierten Signale werden in einem Diagramm kombiniert.
fig, ax = plt.subplots()
ax.plot(sequenz_tabelle["t"], sequenz_tabelle["r_t"], marker="o", label="Reset-Signal $r_t$")
ax.plot(sequenz_tabelle["t"], sequenz_tabelle["z_t"], marker="s", label="Update-Signal $z_t$")
ax.set_ylim(0.0, 1.0)
ax.set_xticks(sequenz_tabelle["t"])
ax.set_xlabel("Zeitschritt t")
ax.set_ylabel("Signalwert")
ax.set_title("Vorgegebene Reset- und Update-Signale über die Zeit")
ax.legend()
plt.tight_layout()
plt.show()


## Plot 3 – Kandidatenzustand

Der Kandidat $g_t$ enthält bereits die durch das Reset-Signal gefilterte Erinnerung, aber noch nicht die Mischung des Update-Signals. Seine separate Darstellung verhindert eine falsche Gleichsetzung von Kandidatenbildung und endgültigem Hidden State.


In [ ]:
# Der Kandidat visualisiert das Ergebnis der Eingabe plus resetteter Erinnerung.
fig, ax = plt.subplots()
ax.plot(
    sequenz_tabelle["t"],
    sequenz_tabelle["g_t"],
    marker="o",
    color="#F58518",
    label=r"Kandidat $g_t$",
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(sequenz_tabelle["t"])
ax.set_xlabel("Zeitschritt t")
ax.set_ylabel("Kandidatenwert")
ax.set_title("Kandidat g_t nach Anwendung des Reset-Signals")
ax.legend()
plt.tight_layout()
plt.show()


## Plot 4 – Resultierender Hidden State

Der Hidden State ist die tatsächlich weitergereichte Erinnerung. Besonders gut sichtbar sind der starke negative Wechsel bei $t=5$ und die Rückkehr ins Positive bei $t=7$, jeweils in Phasen mit kleinem $z_t$: Dann ist $1-z_t$ groß und der neue Kandidat wird stark übernommen.


In [ ]:
# Der Endzustand jedes Schritts wird im nächsten Schritt zur alten Erinnerung.
fig, ax = plt.subplots()
ax.plot(
    sequenz_tabelle["t"],
    sequenz_tabelle["h_t"],
    marker="o",
    color="#54A24B",
    label="Hidden State $h_t$",
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(sequenz_tabelle["t"])
ax.set_xlabel("Zeitschritt t")
ax.set_ylabel("Hidden-State-Wert")
ax.set_title("Resultierender Hidden State über die Zeit")
ax.legend()
plt.tight_layout()
plt.show()


## Plot 5 – Alter Zustand und Kandidat als getrennte Beiträge

Die Balken zerlegen jeden neuen Hidden State in $z_t h_{t-1}$ und $(1-z_t)g_t$. Getrennte, nebeneinanderliegende Balken sind bei positiven und negativen Werten leichter zu lesen als ein Stapel. Ihre rechnerische Summe entspricht jeweils dem Wert im vorherigen Hidden-State-Plot.


In [ ]:
# Nebeneinanderliegende Beiträge machen das Mischen durch das Update-Signal sichtbar.
positionen = sequenz_tabelle["t"].to_numpy()
breite = 0.36
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    positionen - breite / 2,
    sequenz_tabelle["z_t * h_(t-1)"],
    breite,
    label=r"alter Anteil $z_t h_{t-1}$",
    color="#4C78A8",
)
ax.bar(
    positionen + breite / 2,
    sequenz_tabelle["(1 - z_t) * g_t"],
    breite,
    label=r"neuer Anteil $(1-z_t)g_t$",
    color="#F58518",
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(positionen)
ax.set_xlabel("Zeitschritt t")
ax.set_ylabel("Beitrag zu $h_t$")
ax.set_title("Update-Signal: Beiträge von altem Zustand und Kandidat g_t")
ax.legend()
plt.tight_layout()
plt.show()


## GRU-Eingaben mit mehreren Merkmalen

Eine GRU kann pro Zeitschritt statt eines Skalars einen ganzen Merkmalsvektor erhalten, zum Beispiel

$$x_t=[\mathrm{Temperatur}_t,\ \mathrm{Luftfeuchtigkeit}_t].$$

Eine Raumklimasequenz mit acht Zeitschritten und zwei Merkmalen hat die Form `(8, 2)`, mit Batchdimension `(1, 8, 2)`. Die GRU verarbeitet weiterhin einen Zeitschritt nach dem anderen, erhält pro Schritt aber beide Messwerte gemeinsam.

In einer realen GRU sind Reset-Signal $r_t$, Update-Signal $z_t$, Kandidat $g_t$ und Hidden State $h_t$ normalerweise Vektoren. Ihre Dimension wird durch die Zahl der GRU-Einheiten bestimmt und muss nicht der Zahl der Eingabemerkmale entsprechen. Das skalare Hauptbeispiel zeigt dieselbe Signal-Idee lediglich in leichter lesbarer Form; für diesen Dimensionsbezug ist kein zusätzliches Training erforderlich.


In [ ]:
# Zwei Spalten repräsentieren Temperatur und Luftfeuchtigkeit an jedem Zeitschritt.
raumklima = np.array([
    [21.0, 45.0],
    [21.3, 46.0],
    [21.8, 47.0],
    [21.4, 50.0],
    [20.9, 53.0],
    [21.1, 51.0],
    [21.7, 48.0],
    [21.5, 46.0],
])

# Vorne wird die Batchdimension ergänzt: eine Sequenz, acht Schritte, zwei Merkmale.
raumklima_batch = raumklima.reshape(1, 8, 2)
print(f"raumklima.shape = {raumklima.shape}")
print(f"raumklima_batch.shape = {raumklima_batch.shape}")

assert raumklima.shape == (8, 2)
assert raumklima_batch.shape == (1, 8, 2)


## Abschließende Kernaussagen

1. Das **Reset-Signal beeinflusst die Berechnung des Kandidaten** $g_t$.
2. Ein kleines $r_t$ blendet den vorherigen Hidden State bei der Kandidatenberechnung weitgehend aus. Das war in C und D sichtbar: $r_t\odot h_{t-1}$ war nur $-0{,}030$.
3. Ein großes $r_t$ berücksichtigt den vorherigen Hidden State stark. In A und B gingen $-0{,}570$ in den Kandidatenpfad ein.
4. Das **Update-Signal mischt** alten Hidden State und Kandidat: $h_t=z_t\odot h_{t-1}+(1-z_t)\odot g_t$.
5. Bei der hier verwendeten Konvention bewahrt ein großes $z_t$ hauptsächlich den alten Zustand. Deshalb blieben A ($-0{,}564$) und D ($-0{,}538$) nahe bei $h_{t-1}=-0{,}6$.
6. Ein kleines $z_t$ macht $1-z_t$ groß und übernimmt hauptsächlich $g_t$. Dadurch lagen B ($0{,}080$) und C ($0{,}581$) jeweils nahe an ihren Kandidaten.
7. Reset- und Update-Signal erfüllen unterschiedliche, aufeinanderfolgende Aufgaben: zuerst Kandidatenbildung, danach Zustandsmischung.
8. Ein kleines Reset-Signal führt nicht automatisch dazu, dass der endgültige Hidden State vollständig gelöscht wird. Szenario D und Zeitschritt 6 zeigen genau das.
9. Dafür ist zusätzlich entscheidend, wie $z_t$ und $1-z_t$ alten Zustand und Kandidaten mischen. Die Beitragsdiagramme machen beide Summanden mit ihrem Vorzeichen sichtbar.
10. In echten GRU-Modellen werden $r_t$, $g_t$ und $z_t$ aus Eingaben, Hidden States und Gewichten berechnet; ihre Parameter werden während des Trainings gelernt.
11. Die manuell vorgegebenen Werte für $r_t$ und $z_t$ dienen hier ausschließlich dazu, ihre Wirkung kontrolliert und ohne störende Trainingseffekte sichtbar zu machen.
12. Eine GRU kann pro Zeitschritt auch mehrere Merkmale wie Temperatur und Luftfeuchtigkeit verarbeiten; dann hat die Beispielsequenz etwa die Form `(8, 2)` beziehungsweise `(1, 8, 2)` mit Batchdimension.

Die Tabellen und getrennten Diagramme liefern damit denselben Befund aus zwei Blickwinkeln: Das Reset-Signal formt $g_t$; anschließend bewahrt $z_t$ alte Information, während $1-z_t$ den neuen Kandidaten übernimmt. Der resultierende Hidden State $h_t$ wird an den nächsten Zeitschritt und bei Bedarf an die Ausgabeschicht weitergereicht.
